Import necesarios

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prefect import task, flow
import sqlite3


Prueba de la API de frankfurter

In [ ]:
url = "https://api.frankfurter.dev/v2/rates?from=2025-01-01&to=&to=2026-01-01&quotes=usd,gbp"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)

print(f"Código de estado devuelto: {response.status_code}")


if response.status_code == 200:
    datos = response.json()
    print("Tasas de cambio:")

else:
    print(f"Error en la conexión: {response.status_code}")

print("Extracción de datos completada.")

print(list(datos)[-10:-1])

: 

Función de extraccion de los datos

In [ ]:

@task(retries=3, retry_delay_seconds=10)
def extract_data( url="https://api.frankfurter.dev/v2/rates", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, range_time=["2025-01-01", "2025-12-31"], currencies="usd,gbp", base="eur"):
    """
    Función para extraer datos de la API de Frankfurter con manejo de errores y reintentos.
    
    Parámetros:
    - url: URL de la API.
    - headers: Encabezados HTTP para la solicitud.
    - retries: Número de reintentos en caso de error.
    - delay: Tiempo de espera entre reintentos (en segundos).
    - range_time: Lista con las fechas de inicio y fin para la extracción de datos.
    
    Retorna:
    - Un DataFrame con los datos extraídos o None si falla la extracción.
    """

    complete_url = f"{url}?base={base}&quotes={currencies}&from={range_time[0]}&to={range_time[1]}"
    response = requests.get(complete_url, headers=headers)

    print(f"Código de estado devuelto: {response.status_code}")

    if response.status_code == 200:
        print("Datos extraidos con exito")
        datos = response.json()
        return datos

    else:
        print(f"Error en la conexión {response.status_code}")



: 

Prueba

Limpieza de los datos

In [ ]:
@task
def clean_data(data):

    data_df = pd.DataFrame(data)
    data_df['date'] = pd.to_datetime(data_df['date'])

    print("Datos limpiados con exito")
    return data_df


: 

Funcion de carga en la bbdd

In [ ]:

@task
def load_data(df, db_name="historico_divisas.db", table_name="tasas_cambio"):
    """
    Carga el DataFrame limpio en una base de datos SQLite.
    """
    try:
        conn = sqlite3.connect(db_name)
        df.to_sql(table_name, conn, if_exists='append', index=False)
        
        print(f"Carga exitosa: {len(df)} filas insertadas en la tabla '{table_name}'.")
        
    except Exception as e:
        print(f"Error al cargar los datos en la base de datos: {e}")
        
    finally:

        conn.close()

: 

Orquestamos todo el flujo de trabajo con prefect

In [ ]:
@flow(name="PipeLine-Divisas") 
def main_etl():
    datos = extract_data()
    df = clean_data(datos)
    load_data(df)

: 

In [ ]:
main_etl()

: 